# Lecture 09 — Modules and files
## Worked examples

**Programming · Human-Centered Artificial Intelligence · Università degli Studi di Milano**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramonpeter/programming-hcai-2026/blob/main/lectures/09-files-demo.ipynb)

The concepts are on the slides. This notebook solves the two problems from the lecture,
including the wrong turns.

Both parts write files into the folder the notebook runs in. A cell that starts with
`%%writefile name` saves the rest of the cell as a file instead of running it, and a line
that starts with `!` runs a command in a terminal. Both work in Jupyter and in Colab.

---
## A · Estimating π with random points

### Step 1 · One dart

Throw a point into the unit square. It lands inside the quarter circle when its distance
from the corner is at most 1, which is `x*x + y*y <= 1`. No square root needed.

In [ ]:
import random

x = random.random()
y = random.random()
print(x, y, x*x + y*y <= 1)

Run it a few times. Inside should come up roughly three times in four.

### Step 2 · Many darts

A first attempt. `randint` was the random function of lecture 5, so it is the one that
comes to mind.

In [ ]:
def estimate_pi(n):
    """Estimate pi from n random points in the unit square."""
    inside = 0
    for _ in range(n):
        x = random.randint(0, 1)
        y = random.randint(0, 1)
        if x*x + y*y <= 1:
            inside += 1
    return 4 * inside / n

print(estimate_pi(10_000))

About 3, every time, however many points. More points should give a better answer, so
something is wrong with the points themselves. Look at a few.

In [ ]:
[random.randint(0, 1) for _ in range(10)]

Only 0 and 1. Every dart lands on one of the four corners, three corners are inside, and
4 × 3/4 is 3. `randint` gives whole numbers. We need any number between 0 and 1, which is
`random.random()`.

In [ ]:
def estimate_pi(n):
    """Estimate pi from n random points in the unit square."""
    inside = 0
    for _ in range(n):
        x = random.random()
        y = random.random()
        if x*x + y*y <= 1:
            inside += 1
    return 4 * inside / n

print(estimate_pi(100), estimate_pi(100), estimate_pi(10_000))

Close to π now, and different on every run. For testing we want the same numbers each
time, so fix the seed first.

In [ ]:
random.seed(1)
print(estimate_pi(100))
random.seed(1)
print(estimate_pi(100))

### Step 3 · Into a module

The function works, so it moves into `montecarlo.py`, with a couple of test lines so that
whoever opens the file sees how to use it.

In [ ]:
%%writefile montecarlo.py
"""Estimate pi by throwing random points at a square."""
import random


def estimate_pi(n):
    """Estimate pi from n random points in the unit square."""
    inside = 0
    for _ in range(n):
        x = random.random()
        y = random.random()
        if x*x + y*y <= 1:
            inside += 1
    return 4 * inside / n


random.seed(1)
print("test:", estimate_pi(100))
print("test:", 3.0 < estimate_pi(10_000) < 3.3)

In [ ]:
import montecarlo

We only wanted the function, and the tests ran too. Importing runs the whole file, top to
bottom. The test lines go under the `__main__` check. While we are at it, a second function
to compare the estimate with the real value.

In [ ]:
%%writefile montecarlo.py
"""Estimate pi by throwing random points at a square."""
import math
import random


def estimate_pi(n):
    """Estimate pi from n random points in the unit square."""
    inside = 0
    for _ in range(n):
        x = random.random()
        y = random.random()
        if x*x + y*y <= 1:
            inside += 1
    return 4 * inside / n


def compare(n):
    """Return the estimate from n points and its distance from math.pi."""
    estimate = estimate_pi(n)
    return estimate, abs(estimate - math.pi)


if __name__ == "__main__":
    random.seed(1)
    print("test:", estimate_pi(100))
    print("test:", 3.0 < estimate_pi(10_000) < 3.3)

In [ ]:
import montecarlo
montecarlo.compare(100)

The file has `compare`, and Python says it does not. A program imports each module only
once. The second `import` saw that `montecarlo` was already loaded and reused the old
version, from before the edit.

In a notebook, after changing a module, either restart the kernel or reload it.

In [ ]:
import importlib
importlib.reload(montecarlo)
montecarlo.compare(100)

Quiet on import, and `compare` is there. Run as a program, the file still tests itself.

In [ ]:
!python montecarlo.py

### Step 4 · How good is it?

In [ ]:
random.seed(1)
for n in [100, 10_000, 1_000_000]:
    estimate, error = montecarlo.compare(n)
    print(f"{n:>9}  {estimate:.6f}  {error:.6f}")

The million takes a second or so. A hundred times more points gives an error roughly ten
times smaller, so every extra correct digit costs a hundred times the work. That is slow
for π, which has better formulas. It is the only option for many problems that have no
formula at all, and then the same few lines do the job.

**Try it:** change the seed and run the table again. Which line changes most?

---
## B · A week of bike counts

### Step 1 · The data

In real life the file comes from the city. Here the notebook writes it, so that the
example also works in Colab.

In [ ]:
%%writefile bikes.csv
date,counter,count
2026-10-12,Bovisa,412
2026-10-12,Navigli,655
2026-10-12,Sempione,530
2026-10-13,Bovisa,438
2026-10-13,Navigli,702
2026-10-13,Sempione,561
2026-10-14,Bovisa,401
2026-10-14,Navigli,689
2026-10-14,Sempione,549
2026-10-15,Bovisa,455
2026-10-15,Navigli,n/a
2026-10-15,Sempione,580
2026-10-16,Bovisa,390
2026-10-16,Navigli,640
2026-10-16,Sempione,602
2026-10-17,Bovisa,210
2026-10-17,Navigli,512
2026-10-17,Sempione,344
2026-10-18,Bovisa,175
2026-10-18,Navigli,498
2026-10-18,Sempione,301

Before the code, the plan from the slide. Two dictionaries.

- `totals`: key a counter name, value the number of bikes it saw all week.
- `by_day`: key a date, value the number of bikes over all counters that day.

Both are the counting pattern of lecture 7, with a number added instead of 1.

### Step 2 · Totals per counter

In [ ]:
import csv

totals = {}
with open("bikes.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        name = row["counter"]
        totals[name] = totals.get(name, 0) + row["count"]

`0 + "412"`. Every value from a CSV file is a string, even when it looks like a number.
Convert it.

In [ ]:
totals = {}
with open("bikes.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        name = row["counter"]
        totals[name] = totals.get(name, 0) + int(row["count"])

Further this time, and then `n/a`. Which line? Add a print and look, or count: it is the
Navigli counter on the 15th. One bad value stops the whole week.

The error we expect is a `ValueError`, raised by `int` and nothing else. So the `try`
goes around that one line, and the bad lines get counted rather than forgotten.

In [ ]:
totals = {}
skipped = 0
with open("bikes.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        try:
            count = int(row["count"])
        except ValueError:
            skipped += 1
            continue
        name = row["counter"]
        totals[name] = totals.get(name, 0) + count

print(totals, skipped)

`continue` from lecture 4 jumps to the next row. The adding happens only for rows that
converted.

### Step 3 · The busiest day

The same loop can fill `by_day` too. Then find its largest value, with the
best-so-far pattern from lecture 4.

In [ ]:
totals = {}
by_day = {}
skipped = 0
with open("bikes.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        try:
            count = int(row["count"])
        except ValueError:
            skipped += 1
            continue
        totals[row["counter"]] = totals.get(row["counter"], 0) + count
        by_day[row["date"]] = by_day.get(row["date"], 0) + count

best = None
for day in by_day:
    if best is None or by_day[day] > by_day[best]:
        best = day

print(by_day)
print(best, by_day[best])

The 13th, with 1701 bikes. Look at the 15th though: 1035, as quiet as the weekend. That
is not true. One counter is missing that day. Skipping a line is not the same as a count
of zero, and a report that says it skipped one line lets the reader see that.

### Step 4 · Functions, then the report

The cells above are a script. Lecture 8 says: name the steps. One function reads, one
finds the busiest day, one writes, and `main` reads like the plan.

In [ ]:
def read_counts(path):
    """Return the totals per counter, the totals per day, and the number of bad lines."""
    totals = {}
    by_day = {}
    skipped = 0
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            try:
                count = int(row["count"])
            except ValueError:
                skipped += 1
                continue
            totals[row["counter"]] = totals.get(row["counter"], 0) + count
            by_day[row["date"]] = by_day.get(row["date"], 0) + count
    return totals, by_day, skipped


def busiest(by_day):
    """Return the day with the largest count."""
    best = None
    for day in by_day:
        if best is None or by_day[day] > by_day[best]:
            best = day
    return best


def write_report(path, totals, day, skipped):
    """Write the report to the file path, replacing what was there."""
    with open(path, "w", encoding="utf-8") as f:
        f.write("Bike counts, one week")
        for name in totals:
            f.write(f"{name}: {totals[name]}")
        f.write(f"Busiest day: {day}")
        f.write(f"Lines skipped: {skipped}")


def main():
    totals, by_day, skipped = read_counts("bikes.csv")
    write_report("report.txt", totals, busiest(by_day), skipped)


main()

No output, so open the file and check what it holds.

In [ ]:
with open("report.txt", encoding="utf-8") as f:
    print(f.read())

All on one line. `write` adds no newline, unlike `print`. Every line needs its own `\n`.

In [ ]:
def write_report(path, totals, day, skipped):
    """Write the report to the file path, replacing what was there."""
    with open(path, "w", encoding="utf-8") as f:
        f.write("Bike counts, one week\n")
        for name in totals:
            f.write(f"{name}: {totals[name]}\n")
        f.write(f"Busiest day: {day}\n")
        f.write(f"Lines skipped: {skipped}\n")


main()
with open("report.txt", encoding="utf-8") as f:
    print(f.read())

Run `main()` twice and the report is still six lines, not twelve. `"w"` replaced the file.
With `"a"` the second run would have added another copy below the first.

**Try it:** call `read_counts("bikes_november.csv")`, a file that does not exist. What
should a report program do then? Change `main` so that it prints one clear sentence
instead of a traceback.

---
## Where we are going

Next lecture: **recursion**. A function that calls itself, and why that is not an endless
loop.